# Fantasy Football Player Breakout Prediction Pipeline

This notebook demonstrates how to use the `PlayerBreakoutPipeline` class to predict which fantasy football players are likely to have breakout seasons.

The pipeline analyzes historical player performance data to identify patterns that predict when players will experience significant fantasy point increases (30%+ increase by default).

In [ ]:
import sys
sys.path.append('../src')

from player_breakout_pipeline import PlayerBreakoutPipeline
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## Basic Usage

Create a pipeline with recent seasons of data and run predictions:

In [ ]:
# Create pipeline with recent seasons (2018-2024)
seasons = list(range(2018, 2025))
pipeline = PlayerBreakoutPipeline(seasons=seasons, breakout_threshold=0.3)

# Run the pipeline to predict 2025 breakouts
predictions, model_results = pipeline.run(predict_season=2025, save_csv=False)

## Model Performance

In [ ]:
print(f"Training AUC: {model_results['train_auc']:.3f}")
print(f"Test AUC: {model_results['test_auc']:.3f}")
print("\nTest Set Classification Report:")
print(model_results['test_report'])

## Feature Importance

Which factors are most predictive of fantasy breakouts?

In [ ]:
# Display feature importance
import matplotlib.pyplot as plt

features = list(model_results['feature_importance'].keys())
importances = list(model_results['feature_importance'].values())

# Sort by importance
sorted_idx = sorted(range(len(importances)), key=lambda i: importances[i], reverse=True)
sorted_features = [features[i] for i in sorted_idx]
sorted_importances = [importances[i] for i in sorted_idx]

# Plot top 10 features
plt.figure(figsize=(10, 6))
plt.barh(sorted_features[:10], sorted_importances[:10])
plt.xlabel('Feature Importance')
plt.title('Top 10 Most Important Features for Predicting Fantasy Breakouts')
plt.tight_layout()
plt.show()

# Print top features
print("Top 10 Feature Importances:")
for feature, importance in zip(sorted_features[:10], sorted_importances[:10]):
    print(f"{feature}: {importance:.3f}")

## Predictions Analysis

Let's examine the players predicted to have the highest breakout potential:

In [ ]:
print("Top 20 Predicted Breakouts for 2025:")
print(predictions[['player_name', 'position', 'recent_team', 'fantasy_points', 'breakout_probability', 'potential_tier']].head(20))

## Potential Tier Analysis

Break down predictions by potential tier:

In [ ]:
# Potential tier breakdown
print("Potential Tier Distribution:")
print(predictions['potential_tier'].value_counts())
print()

# Position breakdown by potential
print("Potential by Position:")
potential_by_position = predictions.groupby(['position', 'potential_tier']).size().unstack(fill_value=0)
print(potential_by_position)
print()

# High potential players by position
high_potential = predictions[predictions['potential_tier'] == 'High Potential']
print(f"High Potential Players by Position:")
print(high_potential['position'].value_counts())

## Fantasy Point Analysis

Look at the fantasy point distribution for high-potential players:

In [ ]:
# Plot fantasy points vs breakout probability
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.scatter(predictions['fantasy_points'], predictions['breakout_probability'], alpha=0.6)
plt.xlabel('2024 Fantasy Points')
plt.ylabel('Breakout Probability')
plt.title('Fantasy Points vs Breakout Potential')

plt.subplot(1, 2, 2)
potential_tiers = ['Low Potential', 'Medium Potential', 'High Potential']
colors = ['red', 'yellow', 'green']
for tier, color in zip(potential_tiers, colors):
    tier_data = predictions[predictions['potential_tier'] == tier]
    plt.hist(tier_data['fantasy_points'], alpha=0.6, label=tier, color=color, bins=20)
plt.xlabel('2024 Fantasy Points')
plt.ylabel('Count')
plt.title('Fantasy Points Distribution by Potential Tier')
plt.legend()

plt.tight_layout()
plt.show()

## Position-Specific Analysis

Look at high-potential players by position:

In [ ]:
positions = ['QB', 'RB', 'WR', 'TE']
for pos in positions:
    pos_high_potential = high_potential[high_potential['position'] == pos].sort_values('breakout_probability', ascending=False)
    if len(pos_high_potential) > 0:
        print(f"\nTop 5 High-Potential {pos}s:")
        print(pos_high_potential[['player_name', 'recent_team', 'fantasy_points', 'breakout_probability']].head())

## Breakout-Specific Analysis

Analyze what makes players likely to break out:

In [ ]:
# Look at previous fantasy points for high-potential players
print("Previous Fantasy Points Analysis for High-Potential Players:")
print(f"Average 2024 fantasy points for high-potential players: {high_potential['fantasy_points'].mean():.1f}")
print(f"Median 2024 fantasy points for high-potential players: {high_potential['fantasy_points'].median():.1f}")
print(f"Range: {high_potential['fantasy_points'].min():.1f} - {high_potential['fantasy_points'].max():.1f}")
print()

# Compare to all players
print("Comparison to All Players:")
print(f"Average 2024 fantasy points for all players: {predictions['fantasy_points'].mean():.1f}")
print(f"Median 2024 fantasy points for all players: {predictions['fantasy_points'].median():.1f}")
print()

# Show distribution of previous production levels
print("Distribution of 2024 Fantasy Points for High-Potential Players:")
bins = [0, 50, 100, 150, 200, 300, float('inf')]
labels = ['0-50', '50-100', '100-150', '150-200', '200-300', '300+']
high_potential['fp_range'] = pd.cut(high_potential['fantasy_points'], bins=bins, labels=labels)
print(high_potential['fp_range'].value_counts().sort_index())

## Custom Analysis

You can customize the pipeline parameters:

In [ ]:
# Example: More conservative breakout threshold (50% increase)
conservative_pipeline = PlayerBreakoutPipeline(
    seasons=list(range(2020, 2025)),  # Use more recent data
    breakout_threshold=0.5  # 50% increase threshold
)

conservative_predictions, conservative_results = conservative_pipeline.run(predict_season=2025, save_csv=False)

print(f"Conservative Model Performance:")
print(f"Test AUC: {conservative_results['test_auc']:.3f}")
print(f"\nTop 10 Conservative High-Potential Players:")
print(conservative_predictions[conservative_predictions['potential_tier'] == 'High Potential'].head(10)[['player_name', 'position', 'recent_team', 'breakout_probability']])

## Breakout vs Dropoff Comparison

Compare breakout predictions with dropoff predictions:

In [ ]:
# Import and run dropoff pipeline for comparison
from player_dropoff_pipeline import PlayerDropoffPipeline

dropoff_pipeline = PlayerDropoffPipeline(seasons=seasons, dropoff_threshold=0.2)
dropoff_predictions, dropoff_results = dropoff_pipeline.run(predict_season=2025, save_csv=False, run_backtest=False)

# Merge predictions
comparison = predictions[['player_name', 'position', 'recent_team', 'fantasy_points', 'breakout_probability', 'potential_tier']].merge(
    dropoff_predictions[['player_name', 'dropoff_probability', 'risk_tier']], 
    on='player_name', 
    how='inner'
)

print("Players with Both High Breakout Potential and Low Dropoff Risk:")
ideal_targets = comparison[
    (comparison['potential_tier'] == 'High Potential') & 
    (comparison['risk_tier'] == 'Low Risk')
].sort_values('breakout_probability', ascending=False)

print(ideal_targets[['player_name', 'position', 'recent_team', 'fantasy_points', 'breakout_probability', 'dropoff_probability']].head(10))

print(f"\nFound {len(ideal_targets)} players with high breakout potential and low dropoff risk")

## Save Results

Save predictions to CSV for further analysis:

In [ ]:
# Save to CSV
predictions.to_csv('breakout_predictions_2025.csv', index=False)
print("Predictions saved to 'breakout_predictions_2025.csv'")

# Save just high-potential players
high_potential_players = predictions[predictions['potential_tier'] == 'High Potential']
high_potential_players.to_csv('high_potential_players_2025.csv', index=False)
print(f"Saved {len(high_potential_players)} high-potential players to 'high_potential_players_2025.csv'")

# Save comparison data if available
if 'comparison' in locals():
    comparison.to_csv('breakout_vs_dropoff_comparison_2025.csv', index=False)
    print("Breakout vs dropoff comparison saved to 'breakout_vs_dropoff_comparison_2025.csv'")